In [78]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda

In [79]:
url="https://en.wikipedia.org/wiki/Kochi"

In [80]:
loader = WebBaseLoader(url)

In [81]:
documents=loader.load()

In [82]:
documents[0].page_content

'\n\n\n\nKochi - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nEtymology\n\n\n\n\n\n\n\n\n2\nHistory\n\n\n\n\n\n\n\n\n3\nGeography and climate\n\n\n\n\nToggle Geography and climate subsection\n\n\n\n\n\n3.1\nGeography\n\n\n\n\n\n\n\n\n3.2\nClimate\n\n\n\n\n\n\n\n\n\n\n4\nCivic a

In [83]:
documents[0].page_content=' '.join(documents[0].page_content.split())

In [84]:
documents[0].page_content

'Kochi - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us Contribute HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 Etymology 2 History 3 Geography and climate Toggle Geography and climate subsection 3.1 Geography 3.2 Climate 4 Civic administration Toggle Civic administration subsection 4.1 Municipal finance 4.2 Law and order 4.3 Politics 5 Economy 6 Transport Toggle Transport subsection 6.1 Air 6.2 Road 6.3 Public transport 6.3.1 Road 6.3.2 Rail 6.3.3 Metro 6.3.4 Water 7 Demographics 8 Culture 9 Healthcare 10 Education Toggle Education subsection 10.1 Secondary education 10.2 Higher education 11 Social service organisations 12 Media 13 Sports 14 Navy 15 Sister cities 16 Notable people 17 See also 18 References 19 Further reading

In [85]:
len(documents[0].page_content)

118975

In [86]:
#Text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                            chunk_overlap=200)



In [87]:
chunks = text_splitter.split_documents(documents)

In [88]:
chunks

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Kochi', 'title': 'Kochi - Wikipedia', 'language': 'en'}, page_content='Kochi - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us Contribute HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 Etymology 2 History 3 Geography and climate Toggle Geography and climate subsection 3.1 Geography 3.2 Climate 4 Civic administration Toggle Civic administration subsection 4.1 Municipal finance 4.2 Law and order 4.3 Politics 5 Economy 6 Transport Toggle Transport subsection 6.1 Air 6.2 Road 6.3 Public transport 6.3.1 Road 6.3.2 Rail 6.3.3 Metro 6.3.4 Water 7 Demographics 8 Culture 9 Healthcare 10 Education Toggle Education subsection 10.1 Secondary education 10.2 Higher education 11 Soci

In [89]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

In [90]:
vectorstore=Chroma.from_documents(documents=chunks,
                                  embedding=embeddings,
                                  persist_directory=r"C:\Users\devnk\OneDrive\Desktop\RAG Project\VectorDB")

In [91]:
retriever=vectorstore.as_retriever(search_type='mmr', 
                                   search_kwargs={'lambda_mult':0.5, 'k':4})

In [92]:
chat_history=[]

prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """Answer the user's question using only the provided context.
            If the answer is not in the context, say so.
            
            Context:
            {context}"""
        ),
        ("placeholder","{chat_history}"),
        ("human","{question}")
    ])

In [93]:
chat_LLM=ChatOpenAI(model='gpt-4o-mini',
                    max_tokens=500,
                    )

In [94]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)  #The retriever returns relevant document chunks, not the final answer. 
                                                          #It takes those chunks and combines it into a single text

In [95]:
rag_chain=(
    {
        'context': retriever | format_docs,
        'question':RunnablePassthrough(),
        "chat_history":RunnableLambda(lambda _: chat_history[-6:])
    }
        | prompt
        | chat_LLM
        | StrOutputParser()
    
)

In [96]:
question = "What is it's nickname?"
answer = rag_chain.invoke(question)

In [97]:
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=answer)
])


In [98]:
print(chat_history)

[HumanMessage(content="What is it's nickname?", additional_kwargs={}, response_metadata={}), AIMessage(content='The nickname of Kochi is "Queen of the Arabian Sea."', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
